# Multi-task ResNet18 Attribute Training

Notebook train chung shape/color cho head-tune va last-block fine-tune. Bat Internet trong Kaggle de clone repo, va attach thu muc `data/` co `nih_attribute/`, `splits/`, `processed/`.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/GOx9-P/Multiple-Pill-Recognition-And-Interaction-Safety.git'
BRANCH = 'CV_attribute_ResNet18_NguyenGiaBao'
REPO_DIR = Path('/kaggle/working/Multiple-Pill-Recognition-And-Interaction-Safety')

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO_DIR)], check=True)

print('Repository:', REPO_DIR)
print('Branch:', BRANCH)

In [ ]:
# Kaggle da co PyTorch GPU. Chi cai cac package Python nhe can thiet, khong cai lai torch/torchvision.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'pyyaml==6.0.2', 'scikit-learn==1.5.2', 'matplotlib==3.9.2', 'pandas==2.2.3'
], check=True)

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Hay bat GPU accelerator trong Kaggle Settings truoc khi train.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Tu dong tim DATA_ROOT theo cay data/image_all, data/splits va data/processed.
# Khong hard-code slug vi Kaggle co the thay doi so cap thu muc khi mount dataset.
def find_data_root(input_root: Path = Path('/kaggle/input')) -> Path:
    candidates = []
    for image_all_dir in input_root.rglob('image_all'):
        candidate = image_all_dir.parent
        has_images = (image_all_dir / 'nih_attribute/shape').is_dir() and (image_all_dir / 'nih_attribute/color').is_dir()
        has_splits = (candidate / 'splits/nih_attribute/shape').is_dir() and (candidate / 'splits/nih_attribute/color').is_dir()
        if has_images and has_splits and (candidate / 'processed').is_dir():
            candidates.append(candidate)

    unique_candidates = sorted(set(candidates))
    if len(unique_candidates) == 1:
        return unique_candidates[0]
    if not unique_candidates:
        raise FileNotFoundError(
            'Khong tim thay DATA_ROOT. Dataset can co data/image_all/nih_attribute, '
            'data/splits/nih_attribute va data/processed trong /kaggle/input.'
        )
    raise RuntimeError(
        'Tim thay nhieu DATA_ROOT, hay chi dinh mot path: ' +
        ', '.join(str(path) for path in unique_candidates)
    )

DATA_ROOT = find_data_root()
OUTPUT_ROOT = Path('/kaggle/working/attribute_runs')

HEAD_RUN_ID = 'attr_head_v1'
LAST_RUN_ID = 'attr_last_blocks_v1'
RUN_HEAD_TRAIN = True
RUN_HEAD_TEST = True
RUN_LAST_BLOCKS_TRAIN = True
RUN_LAST_BLOCKS_TEST = True

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(REPO_DIR / 'src'))
print('DATA_ROOT:', DATA_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)

In [ ]:
# Preflight: kiem tra schema CSV, source group leakage va synthetic chi nam o train.
from pill_safety.cv.attribute.training.data_contract import validate_attribute_data

paths = {
    'shape_image_dir': DATA_ROOT / 'image_all/nih_attribute/shape',
    'color_image_dir': DATA_ROOT / 'image_all/nih_attribute/color',
    'label_mapping': DATA_ROOT / 'processed/nih_attribute/label_mapping.json',
    'shape_train_csv': DATA_ROOT / 'splits/nih_attribute/shape/train_combined_crop.csv',
    'shape_val_csv': DATA_ROOT / 'splits/nih_attribute/shape/val_combined_crop.csv',
    'shape_test_csv': DATA_ROOT / 'splits/nih_attribute/shape/test_combined_crop.csv',
    'color_train_csv': DATA_ROOT / 'splits/nih_attribute/color/train_multilabel.csv',
    'color_val_csv': DATA_ROOT / 'splits/nih_attribute/color/val_multilabel.csv',
    'color_test_csv': DATA_ROOT / 'splits/nih_attribute/color/test_multilabel.csv',
}
manifest = validate_attribute_data(paths, verify_images=True)
manifest

In [ ]:
from pill_safety.cv.attribute.training.workflow import (
    calibrate_color_thresholds,
    compare_validation_runs,
    evaluate_test,
    load_config,
    train,
)

HEAD_CONFIG = load_config(REPO_DIR / 'configs/training/attribute_resnet18_head_tune/config.yaml')
LAST_CONFIG = load_config(REPO_DIR / 'configs/training/attribute_resnet18_last_blocks_finetune/config.yaml')

In [ ]:
# Phase 1: ImageNet ResNet18, freeze backbone va train shape_head/color_head.
head_checkpoint = OUTPUT_ROOT / 'attribute_resnet18_head_tune/checkpoints' / f'{HEAD_RUN_ID}_best.pt'
if RUN_HEAD_TRAIN:
    head_result = train(HEAD_CONFIG, str(DATA_ROOT), str(OUTPUT_ROOT), HEAD_RUN_ID)
    head_checkpoint = Path(head_result['checkpoint'])
if not head_checkpoint.is_file():
    raise FileNotFoundError(f'Head checkpoint not found: {head_checkpoint}')
head_result = {'checkpoint': str(head_checkpoint)}
head_result

In [ ]:
# Calibration cua head: quet threshold TREN validation, luu rieng theo checkpoint.
head_threshold_result = calibrate_color_thresholds(HEAD_CONFIG, head_checkpoint, str(DATA_ROOT), str(OUTPUT_ROOT), HEAD_RUN_ID)
head_thresholds = Path(head_threshold_result['path'])
head_threshold_result

In [ ]:
# Test chi la reporting sau khi head checkpoint va threshold da duoc chon bang validation.
if RUN_HEAD_TEST:
    head_test = evaluate_test(HEAD_CONFIG, head_checkpoint, head_thresholds, str(DATA_ROOT), str(OUTPUT_ROOT), HEAD_RUN_ID)
    print(head_test['metrics'])

In [ ]:
# Phase 2: bat buoc nap head best checkpoint, unfreeze layer3/layer4 va hai heads.
last_checkpoint = OUTPUT_ROOT / 'attribute_resnet18_last_blocks_finetune/checkpoints' / f'{LAST_RUN_ID}_best.pt'
if RUN_LAST_BLOCKS_TRAIN:
    last_result = train(LAST_CONFIG, str(DATA_ROOT), str(OUTPUT_ROOT), LAST_RUN_ID, pretrained_override=str(head_checkpoint))
    last_checkpoint = Path(last_result['checkpoint'])
if not last_checkpoint.is_file():
    raise FileNotFoundError(f'Last-block checkpoint not found: {last_checkpoint}')
last_result = {'checkpoint': str(last_checkpoint)}
last_result

In [ ]:
# Calibration va test reporting cua last-block checkpoint.
last_threshold_result = calibrate_color_thresholds(LAST_CONFIG, last_checkpoint, str(DATA_ROOT), str(OUTPUT_ROOT), LAST_RUN_ID)
last_thresholds = Path(last_threshold_result['path'])
if RUN_LAST_BLOCKS_TEST:
    last_test = evaluate_test(LAST_CONFIG, last_checkpoint, last_thresholds, str(DATA_ROOT), str(OUTPUT_ROOT), LAST_RUN_ID)
    print(last_test['metrics'])

In [ ]:
# Chon model theo validation; test khong tham gia quyet dinh.
comparison = compare_validation_runs(
    OUTPUT_ROOT / 'attribute_resnet18_head_tune/metrics' / f'{HEAD_RUN_ID}_val_metrics.json',
    OUTPUT_ROOT / 'attribute_resnet18_last_blocks_finetune/metrics' / f'{LAST_RUN_ID}_val_metrics.json',
    OUTPUT_ROOT / 'attribute_model_selection.json',
)
comparison